# Transaction Linking Evaluation

Scores transaction linking model results against ground truth — **no model inference required**.

**Inputs:**
- `evaluation_data/transaction_link_ground_truth.csv` — human-annotated ground truth
- `evaluation_data/transaction_link_model_results.csv` — model extraction output

**Fields evaluated:**
- `RECEIPT_DATE`, `RECEIPT_DESCRIPTION`, `RECEIPT_TOTAL`
- `BANK_TRANSACTION_DATE`, `BANK_TRANSACTION_DESCRIPTION`, `BANK_TRANSACTION_DEBIT`

Both CSVs share the same format: one row per receipt image, pipe-delimited for multi-receipt pages.

In [ ]:
"""Cell 1: Imports and setup."""

import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.batch_analytics import BatchAnalytics  # noqa: E402
from common.batch_reporting import BatchReporter  # noqa: E402
from common.batch_visualizations import BatchVisualizer  # noqa: E402

from common.evaluation_metrics import (  # noqa: E402
    calculate_field_accuracy_with_method,
    load_ground_truth,
)
from common.field_config import get_document_type_fields  # noqa: E402

print("Imports loaded successfully.")

In [ ]:
"""Cell 2: Load ground truth and model results."""

GROUND_TRUTH_PATH = "evaluation_data/transaction_link_ground_truth.csv"
MODEL_RESULTS_PATH = "evaluation_data/transaction_link_model_results.csv"

ground_truth = load_ground_truth(GROUND_TRUTH_PATH, show_sample=True)
model_results = load_ground_truth(MODEL_RESULTS_PATH, verbose=False)

gt_images = set(ground_truth.keys())
mr_images = set(model_results.keys())

missing_in_gt = mr_images - gt_images
extra_in_gt = gt_images - mr_images

if missing_in_gt:
    print(
        f"Warning: {len(missing_in_gt)} images in model results not found in "
        f"ground truth: {sorted(missing_in_gt)}"
    )
if extra_in_gt:
    print(
        f"Info: {len(extra_in_gt)} ground truth images not in model results "
        f"(will be skipped): {sorted(extra_in_gt)}"
    )

common_images = gt_images & mr_images
print(f"\n{len(common_images)} images to evaluate")

In [ ]:
"""Cell 3: Determine evaluation fields for transaction_link."""

# Fields from field_definitions.yaml for transaction_link
all_fields = get_document_type_fields("transaction_link")
print(f"All transaction_link fields ({len(all_fields)}): {all_fields}")

# Exclude metadata columns that aren't model outputs to score
METADATA_FIELDS = {"DOCUMENT_TYPE", "BANK_STATEMENT_FILE"}
eval_fields = [f for f in all_fields if f not in METADATA_FIELDS]
print(f"\nEvaluation fields ({len(eval_fields)}): {eval_fields}")

In [ ]:
"""Cell 4: Build batch_results from model results CSV."""

batch_results = []
processing_times = []

for image_name in sorted(common_images):
    extracted_data = {
        k: v for k, v in model_results[image_name].items() if k != "image_file"
    }

    batch_results.append(
        {
            "image_name": image_name,
            "document_type": "transaction_link",
            "processing_time": 0.0,
            "prompt_used": "staged_transaction_linking",
            "extracted_data": extracted_data,
        }
    )
    processing_times.append(0.0)

print(f"Built {len(batch_results)} results from model CSV.")

In [ ]:
"""Cell 5: Score all fields for all images."""

for result in batch_results:
    image_name = result["image_name"]
    extracted = result["extracted_data"]
    gt = ground_truth.get(image_name, {})

    field_accuracies = {}
    for field in eval_fields:
        ext_val = extracted.get(field, "NOT_FOUND")
        gt_val = gt.get(field, "NOT_FOUND")
        metrics = calculate_field_accuracy_with_method(
            ext_val, gt_val, field, method="order_aware_f1"
        )
        field_accuracies[field] = {"accuracy": metrics["f1_score"]}

    overall = (
        sum(d["accuracy"] for d in field_accuracies.values()) / len(field_accuracies)
        if field_accuracies
        else 0.0
    )
    matched = sum(1 for d in field_accuracies.values() if d["accuracy"] >= 0.5)

    result["evaluation"] = {
        "overall_accuracy": overall,
        "fields_extracted": len(field_accuracies),
        "fields_matched": matched,
        "total_fields": len(eval_fields),
        "field_accuracies": field_accuracies,
    }

    print(f"{image_name}: {overall:.1%} ({matched}/{len(eval_fields)} fields matched)")
    # Show per-field detail
    for field, acc in field_accuracies.items():
        score = acc["accuracy"]
        status = "PASS" if score >= 1.0 else "PARTIAL" if score >= 0.5 else "FAIL"
        if score < 1.0:
            print(f"  [{status}] {field}: {score:.1%}")
            print(f"    got:      {extracted.get(field, 'NOT_FOUND')}")
            print(f"    expected: {gt.get(field, 'NOT_FOUND')}")

avg_acc = sum(r["evaluation"]["overall_accuracy"] for r in batch_results) / len(
    batch_results
)
print(f"\nBatch Average: {avg_acc:.1%}")

In [ ]:
"""Cell 6: Analytics — DataFrames and summary statistics."""

analytics = BatchAnalytics(batch_results, processing_times)

df_results = analytics.create_results_dataframe()
df_summary = analytics.create_summary_statistics(df_results)
df_fields = analytics.create_field_statistics(df_results)

print("=== Summary Statistics ===")
display(df_summary)

if df_fields is not None:
    print("\n=== Field-Level Statistics ===")
    display(df_fields)

In [ ]:
"""Cell 7: Reporting — Markdown executive summary."""

doc_types_found = dict(Counter(r["document_type"] for r in batch_results))
df_doctype = analytics.create_doctype_statistics(df_results)
timestamp = "transaction_linking_eval"

reporter = BatchReporter(batch_results, processing_times, doc_types_found, timestamp)
md_report = reporter.generate_executive_summary(df_results, df_doctype, Path("output"))
print(md_report)

In [ ]:
"""Cell 8: Visualizations — dashboard charts."""

visualizer = BatchVisualizer()

visualizer.create_dashboard(
    df_results,
    df_doctype,
    timestamp,
    save_path=None,
    show=True,
)

visualizer.create_field_heatmap(
    df_results,
    timestamp,
    save_path=None,
    show=True,
)